In [122]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [123]:
import os
from pathlib import Path

import polars as pl

pl.Config(set_tbl_cols=1000)

In [124]:
data_dir = Path(os.path.abspath("")).resolve().parents[1] / "data"
data_files = tuple(data_dir.glob("*.csv"))
data_files

(PosixPath('/home/runar/github/TDT4225-ex3/data/credits.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/keywords.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/links.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/links_small.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/movies_metadata.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/ratings.csv'),
 PosixPath('/home/runar/github/TDT4225-ex3/data/ratings_small.csv'))

In [125]:
from polars.exceptions import ComputeError

for file in data_files:
    try:
        pl.read_csv(file)
    except ComputeError:
        print(f"{file} had problems parsing")

/home/runar/github/TDT4225-ex3/data/movies_metadata.csv had problems parsing


There's a problem in movies_metadata.csv. Some newlines have not been escaped. Fix implemented in `read_csv_merging_split_rows`:

In [126]:
from tdt4225_ex3.data_cleaning import read_csv_merging_split_rows

try:
    read_csv_merging_split_rows(data_dir / "movies_metadata.csv")
    print("Loading fixed")
except ComputeError:
    print("Loading is still failing")

Loading fixed


In [127]:
from tdt4225_ex3.data_cleaning import get_data

datasets = get_data()

Duplication checks and fixes:

In [128]:
for name, df in datasets.items():
    print(f"{name}: {df.height} rows, {df.height - df.unique().height} identical rows")

credits: 45476 rows, 37 identical rows
keywords: 46419 rows, 987 identical rows
links_small: 9125 rows, 0 identical rows
links: 45843 rows, 0 identical rows
movies_metadata: 45463 rows, 17 identical rows
ratings_small: 100004 rows, 0 identical rows
ratings: 26024289 rows, 0 identical rows


In [129]:
for name, df in datasets.items():
    if "id" not in df.columns:
        continue
    df = df.unique()
    print(f"{name}: {df.height - df['id'].n_unique()} duplicate ids")

credits: 7 duplicate ids
keywords: 0 duplicate ids
movies_metadata: 13 duplicate ids


In [130]:
def see_diff_cols_per_id(df: pl.DataFrame) -> pl.DataFrame:
    dups = df.filter(pl.col("id").is_duplicated())
    n = dups.group_by("id").agg(pl.all().n_unique()).drop("id").max()
    cols = [c for c in n.columns if n[c][0] > 1]
    return dups.select("id", *cols).sort("id")


print(see_diff_cols_per_id(datasets["movies_metadata"]))
print(see_diff_cols_per_id(datasets["credits"]))

shape: (59, 3)
┌────────┬────────────┬────────────┐
│ id     ┆ popularity ┆ vote_count │
│ ---    ┆ ---        ┆ ---        │
│ i64    ┆ f64        ┆ i64        │
╞════════╪════════════╪════════════╡
│ 4912   ┆ 11.331072  ┆ 281        │
│ 4912   ┆ 7.645827   ┆ 281        │
│ 5511   ┆ 9.091288   ┆ 187        │
│ 5511   ┆ 9.091288   ┆ 187        │
│ 10991  ┆ 10.264597  ┆ 143        │
│ …      ┆ …          ┆ …          │
│ 168538 ┆ 1.276602   ┆ 3          │
│ 265189 ┆ 12.165685  ┆ 255        │
│ 265189 ┆ 12.165685  ┆ 255        │
│ 298721 ┆ 2.535419   ┆ 50         │
│ 298721 ┆ 2.535419   ┆ 50         │
└────────┴────────────┴────────────┘
shape: (87, 2)
┌────────┬─────────────────────────────────┐
│ id     ┆ crew                            │
│ ---    ┆ ---                             │
│ i64    ┆ str                             │
╞════════╪═════════════════════════════════╡
│ 3057   ┆ [{'credit_id': '52fe4380c3a368… │
│ 3057   ┆ [{'credit_id': '52fe4380c3a368… │
│ 4912   ┆ [{'credit_id': 

In [131]:
for col in ("budget", "revenue", "runtime", "vote_count"):
    print(
        f"{col}: {(datasets['movies_metadata'][col] == 0).sum()} zeros out of {datasets['movies_metadata'].height}"
    )

budget: 36573 zeros out of 45463
revenue: 38055 zeros out of 45463
runtime: 1558 zeros out of 45463
vote_count: 2899 zeros out of 45463


- All:
  - Completely duplicated rows are removed.
- Metadata:
  - Only popularity and vote_count differ per id, so we keep the most votes and highest popularity.
  - Use null where there is 0 for budget, revenue and runtime.
  - Use null for vote_average when vote_count is 0
- Credits:
  - We keep the one with longest crew-list.

This is taken care of in `get_cleaned_data`.

In [132]:
from tdt4225_ex3.data_cleaning import get_cleaned_data

datasets = get_cleaned_data()

By looking at the raw csv files, we see that some columns are json objects:

- credits.csv: cast, crew
- keywords.csv: keywords
- movies_metadata.csv: belongs_to_collection, genres, production_companies, production_countries, spoken_languages

We want to see if there is a consistent schema:

In [133]:
json_columns = {
    "credits": ("cast", "crew"),
    "keywords": ("keywords",),
    "movies_metadata": (
        "belongs_to_collection",
        "genres",
        "production_companies",
        "production_countries",
        "spoken_languages",
    ),
}

In [134]:
import ast
from collections import Counter

for name, cols in json_columns.items():
    for col in cols:
        layouts = Counter()
        for value in datasets[name][col].drop_nulls():
            parsed = ast.literal_eval(value)
            if isinstance(parsed, dict):
                parsed = [parsed]
            for d in parsed:
                layouts[str({k: type(v).__name__ for k, v in sorted(d.items())})] += 1

        print(f"{name}.{col}")
        for layout, count in layouts.most_common():
            print(count, layout)
        print()

credits.cast
388366 {'cast_id': 'int', 'character': 'str', 'credit_id': 'str', 'gender': 'int', 'id': 'int', 'name': 'str', 'order': 'int', 'profile_path': 'str'}
173678 {'cast_id': 'int', 'character': 'str', 'credit_id': 'str', 'gender': 'int', 'id': 'int', 'name': 'str', 'order': 'int', 'profile_path': 'NoneType'}

credits.crew
368835 {'credit_id': 'str', 'department': 'str', 'gender': 'int', 'id': 'int', 'job': 'str', 'name': 'str', 'profile_path': 'NoneType'}
95001 {'credit_id': 'str', 'department': 'str', 'gender': 'int', 'id': 'int', 'job': 'str', 'name': 'str', 'profile_path': 'str'}

keywords.keywords
156602 {'id': 'int', 'name': 'str'}

movies_metadata.belongs_to_collection
3235 {'backdrop_path': 'str', 'id': 'int', 'name': 'str', 'poster_path': 'str'}
710 {'backdrop_path': 'NoneType', 'id': 'int', 'name': 'str', 'poster_path': 'str'}
517 {'backdrop_path': 'NoneType', 'id': 'int', 'name': 'str', 'poster_path': 'NoneType'}
26 {'backdrop_path': 'str', 'id': 'int', 'name': 'str',

If we treat None as empty string (which is reasonable), we see that all the columns have a consistent schema!